In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 11.9 MB/s eta 0:00:00


In [4]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
google_gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051

In [5]:
import os
from pyngrok import ngrok

In [6]:
ngrok.kill()

In [7]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://reawake-brilliant-favorable.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://reawake-brilliant-favorable.ngrok-free.dev


True

In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)

from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    print("EVENT: ", event)
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        line_bot_api.reply_message_with_http_info(
            ReplyMessageRequest(
                reply_token=event.reply_token,
                messages=[TextMessage(text=event.message.text),
                            TextMessage(text=event.message.text)]
            )
        )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit


BODY:  {"destination":"Uecb5c0e1eef83517d8d70393a19967b3","events":[{"type":"message","message":{"type":"text","id":"616036881276600934","quoteToken":"NAvtxwqszmV1yAup5CU7V9zxJLY0raxFFprvEy14d_zXVt15dMngKG2d84MS5HMyrmf4gSTJdwG0fYBxerkDINAIEc2oSmzIDaRQlBN01F-o552pE5D3TwW3pQmvC5RmgyotticDqrULsPJAgWC20A","markAsReadToken":"nXo336b3r4nJEURdWuKX24wAPTkOFCZx_9kBrAoqpTPm79vSxUOlxkS-Ssl6WS-RyjQad1-HBjR_XC0yQL1LhtvmJW1Np4Fso66CY4NEHrDSZtiZUufRRc4BDfS9WxvGtD1yGfibrPyOCBA0Acz76SP4X9RiqcLphz4TyBvmHVSKfdMz0ijrZtH6q_XxIc_0s9EW2eO316oOyPH5_nUlvA","text":"我是誰"},"webhookEventId":"01KSRNB9EEXZ5G8PECSK2DNZWW","deliveryContext":{"isRedelivery":false},"timestamp":1780017898806,"source":{"type":"user","userId":"Uad9bd22bc745d827475a66a9a23dd568"},"replyToken":"cb59043a82e344c7a87d64c0466e09fc","mode":"active"}]}
EVENT:  type='message' source=UserSource(type='user', user_id='Uad9bd22bc745d827475a66a9a23dd568') timestamp=1780017898806 mode=<EventMode.ACTIVE: 'active'> webhook_event_id='01KSRNB9EEXZ5G8PECSK2DN

INFO:werkzeug:127.0.0.1 - - [29/May/2026 01:24:59] "POST / HTTP/1.1" 200 -
